# Texture Synthesis

This notebook explores feature statistics for texture synthesis: channel means and variances, Gram matrices, and pixel-space optimization. The reference image is read from the repository's local filtering examples; a regular checker pattern is generated in the notebook for a failure-case experiment.

The experiments are in progress. The VGG19 feature extractor uses torchvision's pretrained weights.


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()), Path.cwd())
DATA_ROOT = REPO_ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms

# The first run may fetch torchvision's pretrained VGG19 weights if they are not cached.


## Feature statistics


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)
unloader = transforms.ToPILImage()

def load_image(image_path, transform):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)
    return image.to(device)

def unload_image(tensor, title=None):
    image = tensor.cpu().clone().squeeze(0)
    image = unloader(image)
    return image

### Gram matrix


In [ ]:
def get_vgg_gram_matrix(input):
    pass

def get_vgg_mean(input):
    pass

def get_vgg_var(input):
    pass

class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        pass

    def forward(self, img):
        # Return the normalized image based on VGG19 mean and variance
        pass

class StyleFeaturesExtractor:
    def __init__(self, cnn, style_img, normalization_mean, normalization_std, num_layers=10):
        """
        A class to extract style features from a pre-trained VGG19

        Attributes:
        -----------
        cnn : torch.nn.Module -- pre-trained VGG19

        style_img : torch.Tensor

        normalization_mean : torch.Tensor
            The mean tensor for normalizing the input image.

        normalization_std : torch.Tensor
            The standard deviation tensor for normalizing the input image.

        num_layers : int, optional
            The number of convolutional layers from which style features will be extracted.
            Default is 10.

        model : torch.nn.Sequential
            The model built up to the specified number of layers, including normalization.
            This model will be used for extracting features.

        style_features : dict
            A dictionary to store the extracted style features from different VGG layers.
        """

        self.cnn = cnn
        self.style_img = style_img
        self.normalization_mean = normalization_mean
        self.normalization_std = normalization_std
        self.num_layers = num_layers
        self.model = None
        self.style_features = {}
        self.build_model_and_extract_features()

    def build_model_and_extract_features(self):
        """Extract and cache style statistics from an image at selected VGG19 layers."""

        normalization = Normalization(self.normalization_mean, self.normalization_std).to(device)
        model = nn.Sequential()
        model.add_module(str(-1), normalization)
        style_features = {}

        self.model = model
        self.style_features = style_features

## Pixel optimization


In [ ]:
class StyleLoss(nn.Module):
    def __init__(self, target_feature, loss_method="gram"):
        super(StyleLoss, self).__init__()
        self.loss_method = loss_method
        self.loss = 0

        if self.loss_method == "gram":
            self.target = target_feature['gram'].detach()
        elif self.loss_method == "mean_var":
            self.target_mean = target_feature['mean'].detach()
            self.target_var = target_feature['var'].detach()
        else:
            raise ValueError("Invalid loss_method. Choose 'gram' or 'mean_var'.")

    def forward(self, input):
        # Depending on self.loss_method (either "gram" or "mean_var") compute the loss of input features statistics to that of the target
        # Store the computed loss in self.loss
        # Do NOT change input. Simply use it to compute stats and the loss accordingly.

        # self.loss = ?

        pass

        return input

In [ ]:
def get_input_optimizer(input_img):
    optimizer = optim.LBFGS([input_img.requires_grad_()])
    return optimizer

def get_style_model_and_losses(
    style_features_extractor,
    loss_method="gram",
):
    """
    Builds a new model based on the style features extracted from the pre-trained VGG.
    It replaces ReLU layers with out-of-place ReLU operations and adds style loss layers
    based on the features extracted from the style image (TODO).

    Parameters:
    -----------
    style_features_extractor : StyleFeaturesExtractor
        An instance of `StyleFeaturesExtractor` that contains the truncated model and
        extracted style features for each layer.

    loss_method : str, optional
        The method used for calculating the style loss. Default is "gram".

    Returns:
    --------
    new_model : torch.nn.Sequential
        A sequential model consisting of the layers up to the desired depth, including
        the necessary style loss layers.

    style_losses : list
        A list of `StyleLoss` modules that will be used to compute the style loss during optimization.
    """

    model = style_features_extractor.model
    style_losses = []
    new_model = nn.Sequential()

    for idx, layer in model.named_children():
        idx = int(idx)
        if isinstance(layer, nn.ReLU):
            layer = nn.ReLU(inplace=False)
        new_model.add_module(str(idx), layer)

    return new_model, style_losses


def run_texture_synthesis(
    style_features_extractor,
    input_img,
    num_steps=2000,
    loss_method="gram",
):
    """
    Runs the texture synthesis optimization process by applying style transfer
    based on the extracted style features. This function updates the input image to
    match the style of the reference image. You can use the optimizer provided in get_input_optimizer

    Parameters:
    -----------
    style_features_extractor : StyleFeaturesExtractor
        An instance of the StyleFeaturesExtractor class that contains the model
        and extracted style features.

    input_img : torch.Tensor
        The input image tensor (initially random noise)
        that will be updated during the optimization process.

    num_steps : int, optional
        The number of optimization steps to run. Default is 2000.

    loss_method : str, optional
        The method used for calculating the style loss. Default is "gram".

    Returns:
    --------
    input_img : torch.Tensor
        The input image tensor that has been updated to match the style of the reference image.
    """

    model, style_losses = get_style_model_and_losses(
        style_features_extractor, loss_method=loss_method
    )

    return input_img

## Local reference image


In [ ]:
image_size = 256
loader = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
style_image_path = REPO_ROOT / "data" / "Filtering" / "image.png"
Image.open(style_image_path)


### Mean and variance


In [ ]:
style_img = load_image(style_image_path, loader)
input_img = torch.randn(style_img.data.size(), device=device)
loss_method = "mean_var"


### Gram matrix


In [ ]:
style_img = load_image(style_image_path, loader)
input_img = torch.randn(style_img.data.size(), device=device)

loss_method = "gram"


### Initialization experiment


In [ ]:
style_img = load_image(style_image_path, loader)
input_img = torch.randn(style_img.data.size(), device=device)

loss_method = "gram"


## Regular-pattern example

A checker pattern is generated locally to probe the limitations of the current feature objective.


In [ ]:
pattern_size = 256
coordinates = np.arange(pattern_size)
checker = ((coordinates[:, None] // 16 + coordinates[None, :] // 16) % 2 * 255).astype(np.uint8)
regular_image = Image.fromarray(np.repeat(checker[:, :, None], 3, axis=2))
regular_image


### Baseline synthesis


In [ ]:
style_img = loader(regular_image).unsqueeze(0).to(device)
input_img = torch.randn(style_img.data.size(), device=device)
loss_method = "gram"


### Follow-up experiment


In [ ]:
style_img = loader(regular_image).unsqueeze(0).to(device)
input_img = torch.randn(style_img.data.size(), device=device)
